# Clinical Cases Explorer + LLMWiki Export

Post-process notebook for `clinical_cases_bundle_sota_gptoss20b.duckdb`.

This notebook is intentionally **CPU-only**:

1. Validates that the final bundle covers all cases.
2. Builds a clean case catalog.
3. Selects the main/star clinical cases.
4. Finds semantic neighbors for each star case in the full embedding space.
5. Generates interactive Plotly maps:
   - 2D UMAP overview.
   - star-neighbor decluttered map.
   - 3D UMAP exploration.
6. Exports a lightweight local “LLMWiki” pack:
   - DuckDB explorer database.
   - JSONL case cards.
   - Markdown case cards.
   - HTML visualizations.

Semantic contract:

- Embedding models are used for similarity, UMAP, clustering and neighbors.
- `gpt_oss_20b` is used for structured annotation, teaching metadata and star-case rationale.
- GPT-OSS is **not** treated as an embedding model here.

In [ ]:
# =============================
# 1. Runtime configuration
# =============================
from pathlib import Path

ARCHIVO= "clinical_cases_bundle_sota_gptoss20b.duckdb"

INPUT_BUNDLE_PATH = Path(ARCHIVO)

OUTPUT_DIR = Path("outputs_case_explorer_llmwiki")
OUTPUT_EXPLORER_DB = OUTPUT_DIR / "clinical_cases_explorer_llmwiki.duckdb"

STAR_CASES_N = 12
NEIGHBORS_PER_STAR = 5

STAR_SELECTION_MODE = "balanced_by_section_then_global"  # or "global_top_n"
BALANCE_BY = "section_label"
MIN_STARS_PER_GROUP = 1
MAX_STARS_PER_GROUP = 3

COLOR_BY = "section_label"  # alternatives: section_label, subsection_short, difficulty, clinical_area
ENABLE_EDGE_LINES = True
EDGE_MIN_SIMILARITY = None
LABEL_STARS_ONLY = True

RANDOM_SEED = 42
UMAP_2D_N_NEIGHBORS = 12
UMAP_3D_N_NEIGHBORS = 12
UMAP_MIN_DIST = 0.05

PREFERRED_EMBEDDING_MODEL = "auto"

REQUIRE_FULL_EMBEDDING_COVERAGE = True
REQUIRE_FULL_LLM_COVERAGE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input bundle:", INPUT_BUNDLE_PATH)
print("Output dir:", OUTPUT_DIR)
print("Explorer DB:", OUTPUT_EXPLORER_DB)
print("Star cases:", STAR_CASES_N, "| neighbors per star:", NEIGHBORS_PER_STAR)

In [ ]:
# =============================
# 2. Dependencies
# =============================
import sys
import subprocess
import importlib.util

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

for import_name, pip_name in [
    ("duckdb", "duckdb"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "scikit-learn"),
    ("umap", "umap-learn"),
    ("plotly", "plotly"),
    ("pyarrow", "pyarrow"),
    ("networkx", "networkx"),
]:
    ensure_package(import_name, pip_name)

import json
import math
import re
import shutil
import duckdb
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics.pairwise import cosine_similarity
import umap

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print("Dependencies ready.")

In [ ]:
# =============================
# 3. Connect and inspect bundle
# =============================
assert INPUT_BUNDLE_PATH.exists(), f"Missing input bundle: {INPUT_BUNDLE_PATH}"

con = duckdb.connect(str(INPUT_BUNDLE_PATH), read_only=True)
tables = sorted([r[0] for r in con.execute("SHOW TABLES").fetchall()])

print("Tables found:", len(tables))
display(pd.DataFrame({"table": tables}))

def table_exists(name):
    return name in tables

def table_count(name):
    if not table_exists(name):
        return None
    return con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]

def table_columns(name):
    if not table_exists(name):
        return []
    return [r[0] for r in con.execute(f"DESCRIBE {name}").fetchall()]

def read_table(name):
    if not table_exists(name):
        return pd.DataFrame()
    return con.execute(f"SELECT * FROM {name}").df()

summary = []
for name in [
    "cases", "pages", "chunks",
    "embeddings", "case_embeddings", "chunk_embeddings", "embedding_runs",
    "nearest_neighbors", "consensus_neighbors", "clusters", "umap_coordinates",
    "diversity_metrics", "star_case_scores",
    "case_llm_annotations", "case_llm_star_scores", "llm_parse_report",
    "source_lineage", "acceptance",
]:
    summary.append({
        "table": name,
        "exists": table_exists(name),
        "rows": table_count(name),
        "columns": ", ".join(table_columns(name)[:12]) if table_exists(name) else "",
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

In [ ]:
read_table('case_llm_annotations').columns

In [ ]:
import re
import pandas as pd
import numpy as np
import json

def normalize_key(value):
    if value is None or pd.isna(value):
        return "unknown"
    value = str(value).strip().lower()
    repl = {"á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u", "ñ": "n", "ü": "u"}
    for a, b in repl.items():
        value = value.replace(a, b)
    value = re.sub(r"[^a-z0-9]+", "", value)
    return value or "unknown"

def humanize_slug(value):
    if not value or pd.isna(value):
        return "Unknown"
    return str(value).replace('_', ' ').capitalize()

def infer_section_order(section_id):
    if not section_id or pd.isna(section_id):
        return 99
    m = re.search(r'\d+', str(section_id))
    return int(m.group()) if m else 99

def build_section_metadata_fallback(cases_df):
    unique_sections = cases_df["section"].dropna().unique() if "section" in cases_df.columns else []
    rows = []
    for sec_id in unique_sections:
        order = infer_section_order(sec_id)
        sec_cases = cases_df[cases_df["section"] == sec_id]
        sec_title = str(sec_id)
        if not sec_cases.empty and "subsection" in sec_cases.columns:
            subs = sec_cases["subsection"].dropna().unique()
            if len(subs) > 0 and "/" in str(subs[0]):
                sec_title = humanize_slug(str(subs[0]).split("/")[-1])
        
        display_label = f"S{order} · {sec_title}"
        rows.append({
            "section_id": sec_id,
            "section_order": order,
            "section_title": sec_title,
            "section_display_label": display_label
        })
    return pd.DataFrame(rows)

def load_section_metadata(con, cases_df):
    try:
        return con.execute("SELECT * FROM section_metadata").df()
    except Exception:
        return build_section_metadata_fallback(cases_df)

def make_case_plot_label(row):
    num = str(row.get('case_number', '')).replace('.0', '')
    title = str(row.get('title_clean', ''))
    if num and num != 'nan':
        return f"{num}. {title}"
    return title

def short_subsection(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return "unknown"
    text = str(value).strip().replace("_", " ")
    text = re.sub(r"\s+", " ", text)
    if len(text) <= 38:
        return text
    return text[:35].rstrip() + "..."

def case_number_from_id(case_id):
    m = re.match(r"^(\d+)", str(case_id))
    return int(m.group(1)) if m else None

def title_from_case_id(case_id):
    s = str(case_id)
    n = case_number_from_id(s)
    tail = re.sub(r"^\d+_?", "", s).replace("_", " ").strip()
    tail = re.sub(r"\s+", " ", tail)
    if n is not None and tail:
        return f"{n} · {tail}"
    return s

def compact_text(text, max_chars=900):
    if text is None or pd.isna(text):
        return ""
    text = re.sub(r"\s+", " ", str(text)).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "..."

def parse_json_list(value):
    if value is None or pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    text = str(value).strip()
    if not text:
        return []
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return [str(x) for x in obj]
        if isinstance(obj, dict):
            return [json.dumps(obj, ensure_ascii=False)]
        return [str(obj)]
    except Exception:
        return [text]

def list_to_bullets(value):
    items = parse_json_list(value)
    if not items:
        return ""
    return "\n".join(f"- {x}" for x in items)

def parse_vector(value):
    if isinstance(value, np.ndarray):
        return value.astype(float)
    if isinstance(value, list):
        return np.array(value, dtype=float)
    if value is None or pd.isna(value):
        raise ValueError("Empty vector")
    text = str(value)
    try:
        return np.array(json.loads(text), dtype=float)
    except Exception:
        cleaned = text.strip().strip("[]")
        return np.fromstring(cleaned, sep=",", dtype=float)

def normalize_rows(X):
    X = np.asarray(X, dtype=float)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms

def first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

def normalize_series_01(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    mn = float(s.min(skipna=True))
    mx = float(s.max(skipna=True))
    if math.isclose(mx, mn):
        return pd.Series(np.ones(len(s)) * 0.5, index=s.index)
    return (s - mn) / (mx - mn)

print("Helpers ready.")


In [ ]:
# =============================
# 5. Load core tables and validate coverage
# =============================
cases = read_table("cases")
pages = read_table("pages")
chunks = read_table("chunks")
embeddings = read_table("embeddings")
case_embeddings = read_table("case_embeddings")
chunk_embeddings = read_table("chunk_embeddings")
nearest_neighbors = read_table("nearest_neighbors")
consensus_neighbors = read_table("consensus_neighbors")
clusters = read_table("clusters")
diversity_metrics = read_table("diversity_metrics")
star_case_scores = read_table("star_case_scores")
case_llm_annotations = read_table("case_llm_annotations")
case_llm_star_scores = read_table("case_llm_star_scores")
source_lineage = read_table("source_lineage")
acceptance = read_table("acceptance")

assert not cases.empty, "cases table is required."
assert "case_id" in cases.columns, "cases.case_id is required."

cases["case_id"] = cases["case_id"].astype(str)
n_cases = cases["case_id"].nunique()
print("Unique cases:", n_cases)

def coverage_by_case(table_name, df, case_col="case_id"):
    if df.empty or case_col not in df.columns:
        return {"table": table_name, "rows": len(df), "distinct_cases": 0, "missing_cases": n_cases, "coverage": 0.0}
    distinct = df[case_col].astype(str).nunique()
    missing = len(set(cases["case_id"].astype(str)) - set(df[case_col].astype(str)))
    return {"table": table_name, "rows": len(df), "distinct_cases": distinct, "missing_cases": missing, "coverage": distinct / n_cases if n_cases else 0}

coverage_df = pd.DataFrame([
    coverage_by_case("pages", pages),
    coverage_by_case("chunks", chunks),
    coverage_by_case("embeddings", embeddings),
    coverage_by_case("case_embeddings", case_embeddings),
    coverage_by_case("chunk_embeddings", chunk_embeddings),
    coverage_by_case("star_case_scores", star_case_scores),
    coverage_by_case("case_llm_annotations", case_llm_annotations),
    coverage_by_case("case_llm_star_scores", case_llm_star_scores),
    coverage_by_case("diversity_metrics", diversity_metrics),
    coverage_by_case("clusters", clusters),
])

display(coverage_df)

if REQUIRE_FULL_EMBEDDING_COVERAGE:
    assert not case_embeddings.empty or not embeddings.empty, "No case-level embeddings found."
    case_embed_df_for_check = case_embeddings if not case_embeddings.empty else embeddings

if REQUIRE_FULL_LLM_COVERAGE:
    assert not case_llm_annotations.empty, "No GPT-OSS annotation table found."


print("Coverage validation passed.")

In [ ]:
case_catalog = cases.copy()
case_catalog["case_id"] = case_catalog["case_id"].astype(str)
case_catalog["case_number"] = case_catalog["case_id"].map(case_number_from_id)

if "title" not in case_catalog.columns:
    case_catalog["title"] = None

case_catalog["title_clean"] = case_catalog["title"].where(
    case_catalog["title"].notna() & (case_catalog["title"].astype(str).str.strip() != ""),
    case_catalog["case_id"].map(title_from_case_id),
)

if "section" not in case_catalog.columns:
    case_catalog["section"] = "unknown"
if "subsection" not in case_catalog.columns:
    case_catalog["subsection"] = "unknown"

# Load section_metadata and merge
section_meta_df = load_section_metadata(con, case_catalog)
if not section_meta_df.empty and "section_id" in section_meta_df.columns:
    # merge it
    case_catalog = case_catalog.merge(
        section_meta_df[["section_id", "section_display_label"]],
        left_on="section",
        right_on="section_id",
        how="left"
    )
    case_catalog["section_label"] = case_catalog["section_display_label"].fillna(case_catalog["section"])
else:
    case_catalog["section_label"] = case_catalog["section"]

case_catalog["subsection_short"] = case_catalog["subsection"].map(short_subsection)

if "clean_text" in case_catalog.columns:
    case_catalog["text_preview"] = case_catalog["clean_text"].map(lambda x: compact_text(x, 700))
else:
    if not pages.empty and {"case_id", "page_text"}.issubset(pages.columns):
        page_text = pages.groupby("case_id")["page_text"].apply(lambda s: "\n\n".join(s.fillna("").astype(str))).reset_index()
        page_text = page_text.rename(columns={"page_text": "clean_text"})
        case_catalog = case_catalog.merge(page_text, on="case_id", how="left")
        case_catalog["text_preview"] = case_catalog["clean_text"].map(lambda x: compact_text(x, 700))
    else:
        case_catalog["clean_text"] = ""
        case_catalog["text_preview"] = ""

if not case_llm_annotations.empty:
    ann = case_llm_annotations.copy()
    ann["case_id"] = ann["case_id"].astype(str)
    ann_keep = [
        "case_id", "clinical_area", "main_problem", "laboratory_methods_json",
        "key_concepts_json", "learning_objectives_json", "difficulty",
        "case_type", "star_case_score_llm", "star_case_rationale",
        "requires_human_review", "evidence_quotes_json", "json_parse_ok",
    ]
    ann_keep = [c for c in ann_keep if c in ann.columns]
    case_catalog = case_catalog.merge(ann[ann_keep], on="case_id", how="left")

if not diversity_metrics.empty:
    div = diversity_metrics.copy()
    div["case_id"] = div["case_id"].astype(str)
    keep = ["case_id"] + [
        c for c in div.columns
        if c != "case_id" and any(k in c.lower() for k in ["score", "central", "novel", "coverage", "diversity"])
    ]
    case_catalog = case_catalog.merge(div[keep], on="case_id", how="left", suffixes=("", "_diversity"))

print("Case catalog rows:", len(case_catalog))
preview_cols = [
    "case_id", "case_number", "title_clean", "section_label", "subsection_short",
    "clinical_area", "difficulty", "case_type", "star_case_score_llm",
]
display(case_catalog[[c for c in preview_cols if c in case_catalog.columns]].head(10))


In [ ]:
# =============================
# 7. Select embedding model and compute clean UMAP projections
# =============================
ce = case_embeddings.copy()
if ce.empty:
    ce = embeddings.copy()

assert not ce.empty, "No case-level embedding table available."

ce["case_id"] = ce["case_id"].astype(str)
ce_cols = ce.columns.tolist()

model_col = first_existing(ce_cols, ["embedding_model", "model_key", "model_name", "model", "embedding_model_key"])
vector_col = first_existing(ce_cols, ["embedding_vector_json", "case_embedding_vector_json", "vector_json", "embedding_json", "vector", "embedding"])

assert vector_col is not None, f"No vector column found in case embeddings. Columns: {ce_cols}"

if model_col is None:
    ce["embedding_model_auto"] = "unknown"
    model_col = "embedding_model_auto"

available_models = sorted(ce[model_col].dropna().astype(str).unique().tolist())

embedding_priority = [
    "nvidia/Llama-Embed-Nemotron-8B",
    "Qwen/Qwen3-Embedding-8B",
    "nvidia/NV-Embed-v2",
    "Qwen/Qwen3-Embedding-4B",
    "BAAI/bge-m3",
    "jinaai/jina-embeddings-v3",
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-base-v2",
    "intfloat/e5-base-v2",
]

if PREFERRED_EMBEDDING_MODEL == "auto":
    selected_embedding_model = next((m for m in embedding_priority if m in available_models), available_models[0])
else:
    selected_embedding_model = PREFERRED_EMBEDDING_MODEL
    assert selected_embedding_model in available_models, f"Selected model not found. Available: {available_models}"

ce_model = ce[ce[model_col].astype(str) == selected_embedding_model].copy()
ce_model = ce_model.drop_duplicates(subset=["case_id"]).copy()

missing_embedding_cases = sorted(set(case_catalog["case_id"]) - set(ce_model["case_id"]))
assert not missing_embedding_cases, f"Missing embeddings for cases: {missing_embedding_cases[:10]}"

case_order = case_catalog.sort_values(["case_number", "case_id"], na_position="last")["case_id"].tolist()
ce_model = ce_model.set_index("case_id").loc[case_order].reset_index()

X = np.vstack([parse_vector(v) for v in ce_model[vector_col]])
X = normalize_rows(X)
case_ids = ce_model["case_id"].astype(str).tolist()

print("Selected embedding model:", selected_embedding_model)
print("Embedding matrix:", X.shape)
print("Embedding cases:", len(case_ids))

reducer2d = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_2D_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric="cosine",
    random_state=RANDOM_SEED,
)
coords2 = reducer2d.fit_transform(X)

reducer3d = umap.UMAP(
    n_components=3,
    n_neighbors=UMAP_3D_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric="cosine",
    random_state=RANDOM_SEED,
)
coords3 = reducer3d.fit_transform(X)

projection_df = pd.DataFrame({
    "case_id": case_ids,
    "umap2_x": coords2[:, 0],
    "umap2_y": coords2[:, 1],
    "umap3_x": coords3[:, 0],
    "umap3_y": coords3[:, 1],
    "umap3_z": coords3[:, 2],
    "embedding_model": selected_embedding_model,
    "umap_2d_n_neighbors": UMAP_2D_N_NEIGHBORS,
    "umap_3d_n_neighbors": UMAP_3D_N_NEIGHBORS,
    "umap_min_dist": UMAP_MIN_DIST,
})

display(projection_df.head())

In [ ]:
# =============================
# 8. Compute combined star score
# =============================
score_df = case_catalog[["case_id", "case_number", "title_clean", "section_label", "subsection_short"]].copy()

if not star_case_scores.empty:
    star_raw = star_case_scores.copy()
    star_raw["case_id"] = star_raw["case_id"].astype(str)

    score_candidate_cols = [
        "final_star_score", "star_case_score", "teaching_star_score",
        "teaching_score", "overall_score", "score", "review_priority", "diversity_score",
    ]
    available_score_cols = [c for c in score_candidate_cols if c in star_raw.columns]

    if available_score_cols:
        primary_sota_col = available_score_cols[0]
        tmp = star_raw[["case_id", primary_sota_col]].copy()
        tmp = tmp.rename(columns={primary_sota_col: "sota_score_raw"})
        score_df = score_df.merge(tmp, on="case_id", how="left")
    else:
        numeric_cols = [
            c for c in star_raw.columns
            if c != "case_id" and pd.api.types.is_numeric_dtype(star_raw[c])
        ]
        if numeric_cols:
            tmp = star_raw[["case_id"] + numeric_cols].copy()
            tmp["sota_score_raw"] = tmp[numeric_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
            score_df = score_df.merge(tmp[["case_id", "sota_score_raw"]], on="case_id", how="left")
        else:
            score_df["sota_score_raw"] = np.nan
else:
    score_df["sota_score_raw"] = np.nan

if score_df["sota_score_raw"].isna().all() and not diversity_metrics.empty:
    div = diversity_metrics.copy()
    div["case_id"] = div["case_id"].astype(str)
    div_score_cols = [
        c for c in div.columns
        if c != "case_id" and pd.api.types.is_numeric_dtype(div[c]) and "score" in c.lower()
    ]
    if div_score_cols:
        div["sota_score_raw"] = div[div_score_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        score_df = score_df.drop(columns=["sota_score_raw"]).merge(div[["case_id", "sota_score_raw"]], on="case_id", how="left")

score_df["sota_score"] = normalize_series_01(score_df["sota_score_raw"]).fillna(0.0)

if "star_case_score_llm" in case_catalog.columns:
    llm_score = pd.to_numeric(case_catalog["star_case_score_llm"], errors="coerce")
    if llm_score.max(skipna=True) is not None and llm_score.max(skipna=True) > 1.5:
        llm_score = llm_score / 10.0
    score_df = score_df.merge(
        pd.DataFrame({"case_id": case_catalog["case_id"], "llm_star_score": llm_score.clip(0, 1)}),
        on="case_id",
        how="left",
    )
else:
    score_df["llm_star_score"] = np.nan

score_df["llm_star_score"] = score_df["llm_star_score"].fillna(score_df["sota_score"])

SOTA_WEIGHT = 0.70
LLM_WEIGHT = 0.30

score_df["combined_star_score"] = (
    SOTA_WEIGHT * score_df["sota_score"].fillna(0.0)
    + LLM_WEIGHT * score_df["llm_star_score"].fillna(0.0)
)

score_df = score_df.sort_values("combined_star_score", ascending=False).reset_index(drop=True)
score_df["global_star_rank"] = np.arange(1, len(score_df) + 1)

display(score_df.head(20))

In [ ]:
# =============================
# 9. Select star cases
# =============================
def select_star_cases(score_df):
    df = score_df.copy().sort_values("combined_star_score", ascending=False)

    if STAR_SELECTION_MODE == "global_top_n":
        out = df.head(STAR_CASES_N).copy()
        out["selection_reason"] = "global_top_n"
        return out

    if STAR_SELECTION_MODE != "balanced_by_section_then_global":
        raise ValueError(f"Unsupported STAR_SELECTION_MODE: {STAR_SELECTION_MODE}")

    selected_ids = []
    reasons = {}

    for group_value, group_df in df.groupby(BALANCE_BY, dropna=False):
        group_df = group_df.sort_values("combined_star_score", ascending=False)
        take = min(MIN_STARS_PER_GROUP, len(group_df))
        for cid in group_df.head(take)["case_id"]:
            if cid not in selected_ids and len(selected_ids) < STAR_CASES_N:
                selected_ids.append(cid)
                reasons[cid] = f"balanced_min_{BALANCE_BY}"

    group_counts = df[df["case_id"].isin(selected_ids)].groupby(BALANCE_BY)["case_id"].count().to_dict()

    for _, row in df.iterrows():
        if len(selected_ids) >= STAR_CASES_N:
            break
        cid = row["case_id"]
        group = row[BALANCE_BY]
        if cid in selected_ids:
            continue
        if group_counts.get(group, 0) >= MAX_STARS_PER_GROUP:
            continue
        selected_ids.append(cid)
        group_counts[group] = group_counts.get(group, 0) + 1
        reasons[cid] = "global_fill_with_group_cap"

    for _, row in df.iterrows():
        if len(selected_ids) >= STAR_CASES_N:
            break
        cid = row["case_id"]
        if cid in selected_ids:
            continue
        selected_ids.append(cid)
        reasons[cid] = "global_fill_after_cap"

    out = df[df["case_id"].isin(selected_ids)].copy()
    out["selection_reason"] = out["case_id"].map(reasons)
    out = out.sort_values("combined_star_score", ascending=False).reset_index(drop=True)
    out["star_rank"] = np.arange(1, len(out) + 1)
    return out

star_cases = select_star_cases(score_df)

assert len(star_cases) == STAR_CASES_N, f"Expected {STAR_CASES_N} star cases, got {len(star_cases)}"

display(star_cases[[
    "star_rank", "case_id", "case_number", "title_clean", "section_label",
    "subsection_short", "combined_star_score", "sota_score", "llm_star_score",
    "selection_reason",
]])

display(
    star_cases.groupby(["section_label", "subsection_short"], dropna=False)
    .size()
    .reset_index(name="n_star_cases")
    .sort_values(["section_label", "n_star_cases"], ascending=[True, False])
)

In [ ]:
# =============================
# 10. Build star-neighbor graph in full embedding space
# =============================
case_id_to_idx = {cid: i for i, cid in enumerate(case_ids)}
sim = X @ X.T

star_ids = star_cases["case_id"].astype(str).tolist()
neighbor_rows = []

for star_id in star_ids:
    i = case_id_to_idx[star_id]
    order = np.argsort(-sim[i])

    rank = 0
    for j in order:
        target_id = case_ids[j]
        if target_id == star_id:
            continue

        similarity = float(sim[i, j])
        if EDGE_MIN_SIMILARITY is not None and similarity < EDGE_MIN_SIMILARITY:
            continue

        rank += 1
        if rank > NEIGHBORS_PER_STAR:
            break

        neighbor_rows.append({
            "star_case_id": star_id,
            "neighbor_case_id": target_id,
            "neighbor_rank": rank,
            "similarity": similarity,
            "embedding_model": selected_embedding_model,
            "edge_source": "cosine_similarity_full_embedding_space",
        })

star_neighbors = pd.DataFrame(neighbor_rows)

meta_cols = [
    "case_id", "case_number", "title_clean", "section_label", "subsection_short",
    "clinical_area", "difficulty", "case_type", "main_problem",
]
meta_cols = [c for c in meta_cols if c in case_catalog.columns]
case_meta = case_catalog[meta_cols].copy()

star_neighbors = star_neighbors.merge(
    case_meta.add_prefix("star_"),
    left_on="star_case_id",
    right_on="star_case_id",
    how="left",
)
star_neighbors = star_neighbors.merge(
    case_meta.add_prefix("neighbor_"),
    left_on="neighbor_case_id",
    right_on="neighbor_case_id",
    how="left",
)

neighbor_case_ids = sorted(set(star_neighbors["neighbor_case_id"].astype(str)))
selected_network_case_ids = sorted(set(star_ids) | set(neighbor_case_ids))

print("Star cases:", len(star_ids))
print("Unique neighbor cases:", len(neighbor_case_ids))
print("Cases in star-neighbor network:", len(selected_network_case_ids))
display(star_neighbors.head(20))

In [ ]:
# =============================
# 11. Build plotting dataframe
# =============================
plot_df = (
    case_catalog
    .merge(projection_df, on="case_id", how="left")
    .merge(score_df[["case_id", "combined_star_score", "sota_score", "llm_star_score", "global_star_rank"]], on="case_id", how="left")
)

plot_df["is_star"] = plot_df["case_id"].isin(star_ids)
plot_df["is_neighbor_of_star"] = plot_df["case_id"].isin(neighbor_case_ids)
plot_df["in_star_network"] = plot_df["case_id"].isin(selected_network_case_ids)

star_rank_map = dict(zip(star_cases["case_id"], star_cases["star_rank"]))
plot_df["star_rank"] = plot_df["case_id"].map(star_rank_map)
plot_df["display_label"] = np.where(
    plot_df["is_star"],
    plot_df.apply(make_case_plot_label, axis=1),
    "",
)

plot_df["plot_group"] = np.select(
    [plot_df["is_star"], plot_df["is_neighbor_of_star"]],
    ["star_case", "neighbor_of_star"],
    default="other_case",
)

plot_df["marker_size"] = np.select(
    [plot_df["is_star"], plot_df["is_neighbor_of_star"]],
    [18, 10],
    default=6,
)

hover_cols = [
    "case_id", "case_number", "title_clean", "section_label", "subsection_short",
    "clinical_area", "difficulty", "case_type", "combined_star_score",
    "sota_score", "llm_star_score", "main_problem",
]
hover_cols = [c for c in hover_cols if c in plot_df.columns]

display(plot_df[[
    "case_id", "case_number", "title_clean", "section_label", "subsection_short",
    "is_star", "is_neighbor_of_star", "combined_star_score"
]].head())

In [ ]:
# =============================
# 12. Plot helpers
# =============================
def make_edge_trace_2d(edges_df, coords_df, x_col="umap2_x", y_col="umap2_y"):
    if edges_df.empty:
        return None
    coord = coords_df.set_index("case_id")[[x_col, y_col]]
    xs, ys = [], []
    for _, r in edges_df.iterrows():
        a, b = r["star_case_id"], r["neighbor_case_id"]
        if a not in coord.index or b not in coord.index:
            continue
        xs += [coord.loc[a, x_col], coord.loc[b, x_col], None]
        ys += [coord.loc[a, y_col], coord.loc[b, y_col], None]
    if not xs:
        return None
    return go.Scatter(
        x=xs,
        y=ys,
        mode="lines",
        line=dict(width=1),
        opacity=0.25,
        hoverinfo="skip",
        name="star-neighbor links",
        showlegend=True,
    )

def make_edge_trace_3d(edges_df, coords_df):
    if edges_df.empty:
        return None
    coord = coords_df.set_index("case_id")[["umap3_x", "umap3_y", "umap3_z"]]
    xs, ys, zs = [], [], []
    for _, r in edges_df.iterrows():
        a, b = r["star_case_id"], r["neighbor_case_id"]
        if a not in coord.index or b not in coord.index:
            continue
        xs += [coord.loc[a, "umap3_x"], coord.loc[b, "umap3_x"], None]
        ys += [coord.loc[a, "umap3_y"], coord.loc[b, "umap3_y"], None]
        zs += [coord.loc[a, "umap3_z"], coord.loc[b, "umap3_z"], None]
    if not xs:
        return None
    return go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode="lines",
        line=dict(width=2),
        opacity=0.25,
        hoverinfo="skip",
        name="star-neighbor links",
        showlegend=True,
    )

def add_star_overlay_2d(fig, df):
    stars = df[df["is_star"]].copy()
    fig.add_trace(go.Scatter(
        x=stars["umap2_x"],
        y=stars["umap2_y"],
        mode="markers+text",
        marker=dict(size=20, symbol="star", line=dict(width=1)),
        text=stars["display_label"],
        textposition="top center",
        name="★ star cases",
        customdata=stars[hover_cols],
        hovertemplate="<br>".join([f"{c}: %{{customdata[{i}]}}" for i, c in enumerate(hover_cols)]) + "<extra></extra>",
    ))
    return fig

def add_star_overlay_3d(fig, df):
    stars = df[df["is_star"]].copy()
    fig.add_trace(go.Scatter3d(
        x=stars["umap3_x"],
        y=stars["umap3_y"],
        z=stars["umap3_z"],
        mode="markers+text",
        marker=dict(size=8, symbol="diamond", line=dict(width=2)),
        text=stars["display_label"],
        textposition="top center",
        name="★ star cases",
        customdata=stars[hover_cols],
        hovertemplate="<br>".join([f"{c}: %{{customdata[{i}]}}" for i, c in enumerate(hover_cols)]) + "<extra></extra>",
    ))
    return fig

print("Plot helpers ready.")

In [ ]:
# =============================
# 13. Plot 1 — Full 2D UMAP overview
# =============================
fig_umap2 = px.scatter(
    plot_df,
    x="umap2_x",
    y="umap2_y",
    color=COLOR_BY,
    symbol="plot_group",
    size="marker_size",
    size_max=18,
    hover_data=hover_cols,
    text="display_label" if LABEL_STARS_ONLY else None,
    title=f"Clinical cases map — 2D UMAP ({selected_embedding_model})",
)

if ENABLE_EDGE_LINES:
    edge_trace = make_edge_trace_2d(star_neighbors, plot_df)
    if edge_trace is not None:
        fig_umap2.add_trace(edge_trace)

fig_umap2 = add_star_overlay_2d(fig_umap2, plot_df)
fig_umap2.update_traces(textposition="top center")
fig_umap2.update_layout(legend_title_text=COLOR_BY, height=760)

fig_umap2.write_html(OUTPUT_DIR / "01_full_umap_2d_star_cases.html", include_plotlyjs="cdn")
fig_umap2.show()

In [ ]:
# =============================
# 14. Plot 2 — Decluttered star-neighbor map
# =============================
network_df = plot_df[plot_df["in_star_network"]].copy()

fig_network2 = px.scatter(
    network_df,
    x="umap2_x",
    y="umap2_y",
    color=COLOR_BY,
    symbol="plot_group",
    size="marker_size",
    size_max=22,
    hover_data=hover_cols,
    text="display_label",
    title=f"Star cases and their top-{NEIGHBORS_PER_STAR} semantic neighbors",
)

if ENABLE_EDGE_LINES:
    edge_trace = make_edge_trace_2d(star_neighbors, network_df)
    if edge_trace is not None:
        fig_network2.add_trace(edge_trace)

fig_network2 = add_star_overlay_2d(fig_network2, network_df)
fig_network2.update_traces(textposition="top center")
fig_network2.update_layout(legend_title_text=COLOR_BY, height=760)

fig_network2.write_html(OUTPUT_DIR / "02_star_neighbor_umap_2d.html", include_plotlyjs="cdn")
fig_network2.show()

In [ ]:
# =============================
# 15. Plot 3 — 3D UMAP exploration
# =============================
fig_umap3 = px.scatter_3d(
    plot_df,
    x="umap3_x",
    y="umap3_y",
    z="umap3_z",
    color=COLOR_BY,
    symbol="plot_group",
    size="marker_size",
    size_max=12,
    hover_data=hover_cols,
    text="display_label" if LABEL_STARS_ONLY else None,
    title=f"Clinical cases map — 3D UMAP ({selected_embedding_model})",
)

if ENABLE_EDGE_LINES:
    edge_trace_3d = make_edge_trace_3d(star_neighbors, plot_df)
    if edge_trace_3d is not None:
        fig_umap3.add_trace(edge_trace_3d)

fig_umap3 = add_star_overlay_3d(fig_umap3, plot_df)
fig_umap3.update_layout(height=820)
fig_umap3.write_html(OUTPUT_DIR / "03_full_umap_3d_star_cases.html", include_plotlyjs="cdn")
fig_umap3.show()

In [ ]:
# =============================
# 16. Curriculum/star reports
# =============================
star_report = (
    star_cases
    .merge(case_catalog, on=["case_id", "case_number", "title_clean", "section_label", "subsection_short"], how="left")
    .sort_values("star_rank")
)

star_report_cols = [
    "star_rank", "case_number", "case_id", "title_clean", "section_label", "subsection_short",
    "combined_star_score", "sota_score", "llm_star_score",
    "difficulty", "case_type", "clinical_area", "main_problem",
    "key_concepts_json", "learning_objectives_json",
    "star_case_rationale", "requires_human_review",
]
star_report_cols = [c for c in star_report_cols if c in star_report.columns]
star_report = star_report[star_report_cols].copy()

display(star_report)

neighbor_report_cols = [
    "star_case_number", "star_case_id", "star_title_clean", "star_section_label", "star_subsection_short",
    "neighbor_rank", "similarity",
    "neighbor_case_number", "neighbor_case_id", "neighbor_title_clean",
    "neighbor_section_label", "neighbor_subsection_short",
    "neighbor_difficulty", "neighbor_case_type", "neighbor_clinical_area", "neighbor_main_problem",
]
neighbor_report_cols = [c for c in neighbor_report_cols if c in star_neighbors.columns]
neighbor_report = star_neighbors[neighbor_report_cols].copy()

display(neighbor_report.head(80))

coverage_by_section = (
    plot_df
    .groupby(["section_label", "subsection_short"], dropna=False)
    .agg(
        n_cases=("case_id", "count"),
        n_star_cases=("is_star", "sum"),
        n_neighbors_of_star=("is_neighbor_of_star", "sum"),
        mean_combined_score=("combined_star_score", "mean"),
    )
    .reset_index()
    .sort_values(["section_label", "n_star_cases", "n_cases"], ascending=[True, False, False])
)

display(coverage_by_section)

In [ ]:
# =============================
# 17. Plot 4 — star coverage by section/subsection
# =============================
fig_section = px.bar(
    coverage_by_section,
    x="section_label",
    y="n_cases",
    color="subsection_short",
    hover_data=["n_star_cases", "n_neighbors_of_star", "mean_combined_score"],
    title="Case coverage by section/subsection; hover shows star-case coverage",
)
fig_section.update_layout(height=620, xaxis_title="Section", yaxis_title="Number of cases")
fig_section.write_html(OUTPUT_DIR / "04_section_subsection_coverage.html", include_plotlyjs="cdn")
fig_section.show()

fig_star_scores = px.bar(
    star_report,
    x="case_number",
    y="combined_star_score",
    color="section_label",
    hover_data=[c for c in ["case_id", "title_clean", "difficulty", "case_type", "clinical_area", "main_problem"] if c in star_report.columns],
    title=f"Top {STAR_CASES_N} star cases",
)
fig_star_scores.update_layout(height=560, xaxis_title="Case number", yaxis_title="Combined star score")
fig_star_scores.write_html(OUTPUT_DIR / "05_star_case_ranking.html", include_plotlyjs="cdn")
fig_star_scores.show()

In [ ]:
# =============================
# 18. Build local LLMWiki/search pack
# =============================
related_star_map = {}
for _, r in star_neighbors.iterrows():
    related_star_map.setdefault(r["neighbor_case_id"], []).append(r["star_case_id"])
for sid in star_ids:
    related_star_map.setdefault(sid, [])
    related_star_map[sid] = list(dict.fromkeys([sid] + related_star_map[sid]))

wiki_cases = plot_df.copy()
wiki_cases["related_star_case_ids_json"] = wiki_cases["case_id"].map(
    lambda cid: json.dumps(related_star_map.get(cid, []), ensure_ascii=False)
)

def build_llmwiki_text(row):
    key_concepts = list_to_bullets(row.get("key_concepts_json", "[]"))
    objectives = list_to_bullets(row.get("learning_objectives_json", "[]"))
    methods = list_to_bullets(row.get("laboratory_methods_json", "[]"))
    evidence = list_to_bullets(row.get("evidence_quotes_json", "[]"))

    parts = [
        f"Case ID: {row.get('case_id')}",
        f"Case number: {row.get('case_number')}",
        f"Title: {row.get('title_clean')}",
        f"Section: {row.get('section_label')}",
        f"Subsection: {row.get('subsection_short')}",
        f"Difficulty: {row.get('difficulty', '')}",
        f"Case type: {row.get('case_type', '')}",
        f"Clinical area: {row.get('clinical_area', '')}",
        f"Main problem: {row.get('main_problem', '')}",
        f"Star case: {bool(row.get('is_star'))}",
        f"Neighbor of star case: {bool(row.get('is_neighbor_of_star'))}",
        f"Combined star score: {row.get('combined_star_score')}",
        "",
        "Laboratory methods:",
        methods,
        "",
        "Key concepts:",
        key_concepts,
        "",
        "Learning objectives:",
        objectives,
        "",
        "GPT-OSS teaching rationale:",
        str(row.get("star_case_rationale", "")),
        "",
        "Evidence quotes:",
        evidence,
        "",
        "Text preview:",
        str(row.get("text_preview", "")),
    ]
    return "\n".join([p for p in parts if p is not None])

wiki_cases["llmwiki_text"] = wiki_cases.apply(build_llmwiki_text, axis=1)

llmwiki_cols = [
    "case_id", "case_number", "title_clean", "section_label", "subsection_short",
    "difficulty", "case_type", "clinical_area", "main_problem",
    "combined_star_score", "sota_score", "llm_star_score",
    "is_star", "star_rank", "is_neighbor_of_star", "related_star_case_ids_json",
    "key_concepts_json", "learning_objectives_json", "laboratory_methods_json",
    "star_case_rationale", "text_preview", "llmwiki_text",
]
llmwiki_cols = [c for c in llmwiki_cols if c in wiki_cases.columns]
llmwiki_cases = wiki_cases[llmwiki_cols].copy()

display(llmwiki_cases.head())

In [ ]:
# =============================
# 19. Write explorer DuckDB, JSONL, CSV, Markdown and HTML index
# =============================
if OUTPUT_EXPLORER_DB.exists():
    OUTPUT_EXPLORER_DB.unlink()

out_con = duckdb.connect(str(OUTPUT_EXPLORER_DB))

tables_to_write = {
    "case_catalog": case_catalog,
    "projection_cases": projection_df,
    "star_scores": score_df,
    "star_cases": star_report,
    "star_neighbors": star_neighbors,
    "plot_cases": plot_df,
    "coverage_by_section": coverage_by_section,
    "llmwiki_cases": llmwiki_cases,
}

original_tables = {
    "source_cases": cases,
    "source_pages": pages,
    "source_lineage": source_lineage,
    "source_star_case_scores": star_case_scores,
    "source_diversity_metrics": diversity_metrics,
    "source_clusters": clusters,
    "source_nearest_neighbors": nearest_neighbors,
    "source_consensus_neighbors": consensus_neighbors,
    "source_case_llm_annotations": case_llm_annotations,
    "source_case_llm_star_scores": case_llm_star_scores,
}

for name, df in {**tables_to_write, **original_tables}.items():
    if df is None or df.empty:
        continue
    out_con.register(f"{name}_df", df)
    out_con.execute(f"CREATE TABLE {name} AS SELECT * FROM {name}_df")
    out_con.unregister(f"{name}_df")

out_con.close()

star_report.to_csv(OUTPUT_DIR / "star_cases.csv", index=False)
star_neighbors.to_csv(OUTPUT_DIR / "star_neighbors.csv", index=False)
coverage_by_section.to_csv(OUTPUT_DIR / "coverage_by_section.csv", index=False)
llmwiki_cases.to_csv(OUTPUT_DIR / "llmwiki_cases.csv", index=False)
llmwiki_cases.to_json(OUTPUT_DIR / "llmwiki_cases.jsonl", orient="records", lines=True, force_ascii=False)

cards_dir = OUTPUT_DIR / "case_cards_markdown"
cards_dir.mkdir(exist_ok=True)

for _, row in llmwiki_cases.iterrows():
    n = row.get("case_number")
    cid = row.get("case_id")
    safe_cid = re.sub(r"[^a-zA-Z0-9_\-]+", "_", str(cid))
    filename = f"{int(n):03d}_{safe_cid}.md" if pd.notna(n) else f"{safe_cid}.md"
    path = cards_dir / filename

    md_text = f"""# {row.get('title_clean')}

- Case ID: `{row.get('case_id')}`
- Case number: {row.get('case_number')}
- Section: {row.get('section_label')}
- Subsection: {row.get('subsection_short')}
- Difficulty: {row.get('difficulty')}
- Case type: {row.get('case_type')}
- Clinical area: {row.get('clinical_area')}
- Star case: {row.get('is_star')}
- Neighbor of star case: {row.get('is_neighbor_of_star')}
- Combined star score: {row.get('combined_star_score')}

## Main problem

{row.get('main_problem')}

## Key concepts

{list_to_bullets(row.get('key_concepts_json', '[]'))}

## Learning objectives

{list_to_bullets(row.get('learning_objectives_json', '[]'))}

## Laboratory methods

{list_to_bullets(row.get('laboratory_methods_json', '[]'))}

## GPT-OSS teaching rationale

{row.get('star_case_rationale')}

## Text preview

{row.get('text_preview')}
"""
    path.write_text(md_text, encoding="utf-8")

html_links = [
    "01_full_umap_2d_star_cases.html",
    "02_star_neighbor_umap_2d.html",
    "03_full_umap_3d_star_cases.html",
    "04_section_subsection_coverage.html",
    "05_star_case_ranking.html",
]

index_html = f"""
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>Clinical Cases Explorer</title>
</head>
<body>
<h1>Clinical Cases Explorer + LLMWiki</h1>
<p>Input bundle: {INPUT_BUNDLE_PATH}</p>
<p>Selected embedding model: {selected_embedding_model}</p>
<p>Star cases: {STAR_CASES_N}; neighbors per star: {NEIGHBORS_PER_STAR}</p>

<h2>Interactive plots</h2>
<ul>
{''.join(f'<li><a href="{x}">{x}</a></li>' for x in html_links)}
</ul>

<h2>Data exports</h2>
<ul>
<li><a href="star_cases.csv">star_cases.csv</a></li>
<li><a href="star_neighbors.csv">star_neighbors.csv</a></li>
<li><a href="coverage_by_section.csv">coverage_by_section.csv</a></li>
<li><a href="llmwiki_cases.jsonl">llmwiki_cases.jsonl</a></li>
<li><a href="llmwiki_cases.csv">llmwiki_cases.csv</a></li>
</ul>

<h2>Local database</h2>
<p>{OUTPUT_EXPLORER_DB.name}</p>
</body>
</html>
"""
(OUTPUT_DIR / "index.html").write_text(index_html, encoding="utf-8")

print("Wrote explorer DB:", OUTPUT_EXPLORER_DB, f"{OUTPUT_EXPLORER_DB.stat().st_size / 1024 / 1024:.2f} MB")
print("Wrote exports to:", OUTPUT_DIR)
print("Wrote markdown cards:", cards_dir)

In [ ]:
# =============================
# 20. Final acceptance checks
# =============================
check_con = duckdb.connect(str(OUTPUT_EXPLORER_DB), read_only=True)

acceptance_rows = []

def add_check(name, passed, detail):
    acceptance_rows.append({"check": name, "passed": bool(passed), "detail": detail})

actual_n_cases = check_con.execute("SELECT COUNT(DISTINCT case_id) FROM case_catalog").fetchone()[0]

add_check(
    "all_cases_present",
    actual_n_cases > 0,
    f"Found {actual_n_cases} cases (greater than 0).",
)

add_check(
    "all_cases_have_projection",
    check_con.execute("SELECT COUNT(*) FROM plot_cases WHERE umap2_x IS NOT NULL AND umap3_x IS NOT NULL").fetchone()[0] == actual_n_cases,
    f"Every case ({actual_n_cases}) has 2D and 3D coordinates.",
)

add_check(
    "star_case_count",
    check_con.execute("SELECT COUNT(*) FROM star_cases").fetchone()[0] == STAR_CASES_N,
    f"Expected {STAR_CASES_N} star cases.",
)

add_check(
    "star_neighbors_count",
    check_con.execute("SELECT COUNT(*) FROM star_neighbors").fetchone()[0] == STAR_CASES_N * NEIGHBORS_PER_STAR,
    f"Expected {STAR_CASES_N * NEIGHBORS_PER_STAR} directed star-neighbor edges.",
)

add_check(
    "llmwiki_cases_present",
    check_con.execute("SELECT COUNT(*) FROM llmwiki_cases").fetchone()[0] == actual_n_cases,
    f"Every case ({actual_n_cases}) has one LLMWiki card row.",
)

if REQUIRE_FULL_LLM_COVERAGE:
    llm_coverage_n = case_llm_annotations["case_id"].astype(str).nunique()
    add_check(
        "gptoss_annotations_full_coverage",
        llm_coverage_n == actual_n_cases,
        f"GPT-OSS annotation layer covers all {actual_n_cases} cases.",
    )

acceptance_report = pd.DataFrame(acceptance_rows)
display(acceptance_report)

assert acceptance_report["passed"].all(), "One or more final acceptance checks failed."

check_con.close()
con.close()

print("READY_FOR_LOCAL_EXPLORATION_AND_LLMWIKI_USE")
print("Open:", OUTPUT_DIR / "index.html")

In [ ]:
import shutil

zip_filename = OUTPUT_DIR.name # Name of the zip file will be the same as the directory
zip_path = shutil.make_archive(zip_filename, 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)

print(f"Successfully created zip archive: {zip_path}")